# Kinematics

运动学（Kinematics）是机器人操纵的几何基础，是连接高层任务规划与低层物理执行的关键桥梁。它描述了机器人的运动，而不考虑引起这些运动的力。本章将从运动学的基础概念出发，系统地探讨其理论、求解方法、优化策略，并最终揭示它在现代机器人核心挑战——任务与运动规划（TAMP）框架下的关键作用。

## 什么是运动学？

运动学是研究运动的几何学，它描述物体的位置、速度和加速度，而不考虑导致这些运动的力或质量。在机器人学中，运动学专注于机械臂的连杆和关节的几何关系。它包含了以下几个重要的概念：**构型空间** (Configuration Space, C-Space)，**正向运动学** (Forward Kinematics, FK)，**逆运动学** (Inverse Kinematics, IK)

- **构型空间 (Configuration Space, C-Space)**

构型空间（C-Space）是一个数学概念，代表了机器人所有可能状态的集合。对于一个由 `n` 个独立关节组成的机械臂，其构型可以用一个 `n` 维向量 $q = [q_1, q_2, ..., q_n]^T$ 来描述，这个向量中的每一个元素代表一个关节的位置。这个 `n` 维向量空间就是该机械臂的构型空间。

In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from ipywidgets import interact, widgets
from IPython.display import display

# --- 1. 定义机器人和环境 ---

# 连杆长度
L1 = 1.0
L2 = 1.0

# 机器人基座位置（地图中心）
P0_base = np.array([1.5, 1.5])

# 地图范围
map_range = [0, 3]

# 障碍物列表 (字典格式: 'center' 是 (x, y) 元组, 'r' 是半径)
obstacles = [
    {'center': (0.5, 0.5), 'r': 0.3},
    {'center': (2.0, 2.0), 'r': 0.4},
    {'center': (1.0, 2.5), 'r': 0.2},
    {'center': (2.5, 1.0), 'r': 0.35},
    {'center': (0.7, 1.8), 'r': 0.25},
]

# --- 2. 碰撞检测函数 ---

def check_line_seg_circle_collision(P_a, P_b, C, r):
    """
    检查线段 (P_a, P_b) 是否与圆心为 C、半径为 r 的圆碰撞
    """
    P_a = np.array(P_a)
    P_b = np.array(P_b)
    C = np.array(C)
    
    # 线段向量
    V = P_b - P_a
    # 从 P_a 到圆心的向量
    A_C = C - P_a
    
    # 计算 V 的长度的平方
    V_len_sq = np.dot(V, V)
    if V_len_sq == 0:
        # P_a 和 P_b 是同一个点
        return np.linalg.norm(A_C) < r

    # 计算投影 t
    # t 是 C 在线段 AB 上的投影点 P_proj 满足 P_proj = P_a + t * V
    t = np.dot(A_C, V) / V_len_sq
    
    # 将 t 限制在 [0, 1] 范围内，找到线段上离圆心最近的点
    t = max(0, min(1, t))
    
    closest_point_on_seg = P_a + t * V
    
    # 计算最近点到圆心的距离
    dist = np.linalg.norm(closest_point_on_seg - C)
    
    return dist < r

# --- 3. 正向运动学 (FK) ---

def forward_kinematics(theta1_rad, theta2_rad):
    """
    根据角度计算 P0, P1, P2 的位置
    """
    P0 = P0_base
    P1 = P0 + np.array([L1 * np.cos(theta1_rad), L1 * np.sin(theta1_rad)])
    P2 = P1 + np.array([L2 * np.cos(theta1_rad + theta2_rad), L2 * np.sin(theta1_rad + theta2_rad)])
    return P0, P1, P2

# --- 4. 预计算 C-Space (构型空间) ---

def compute_c_space_grid(resolution=100):
    """
    遍历所有 (t1, t2) 组合，检查碰撞，生成 C-Space 网格
    """
    print("正在计算 C-Space 网格... (可能需要几秒钟)")
    
    # 角度范围
    theta_range = np.linspace(-np.pi, np.pi, resolution)
    # 角度（度）范围，用于绘图
    theta_range_deg = np.rad2deg(theta_range)
    
    # 碰撞网格 (0 = 自由, 1 = 碰撞)
    collision_grid = np.zeros((resolution, resolution))
    
    for i, t1 in enumerate(theta_range):
        for j, t2 in enumerate(theta_range):
            # 1. 计算 FK
            P0, P1, P2 = forward_kinematics(t1, t2)
            
            # 2. 检查碰撞
            collides = False
            for obs in obstacles:
                # 检查第一个连杆
                if check_line_seg_circle_collision(P0, P1, obs['center'], obs['r']):
                    collides = True
                    break
                # 检查第二个连杆
                if check_line_seg_circle_collision(P1, P2, obs['center'], obs['r']):
                    collides = True
                    break
            
            if collides:
                collision_grid[j, i] = 1 # j 对应 t2 (y轴), i 对应 t1 (x轴)
                
    print("C-Space 网格计算完成。")
    return collision_grid, theta_range_deg

# 执行计算
c_space_grid, theta_range_deg = compute_c_space_grid(resolution=100)


# --- 5. 初始化 Plotly 图表 (使用 FigureWidget) ---

# 创建 1x2 子图
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Workspace (工作空间)", "Configuration Space (C-空间)")
)

# === 子图 1: 工作空间 ===

# a. 添加机器人 (初始位置)
P0_init, P1_init, P2_init = forward_kinematics(0, 0)
fig.add_trace(
    go.Scatter(
        x=[P0_init[0], P1_init[0], P2_init[0]],
        y=[P0_init[1], P1_init[1], P2_init[1]],
        mode='lines+markers',
        line=dict(width=5),
        marker=dict(size=10),
        name='Robot'
    ),
    row=1, col=1
)

# b. 添加障碍物 (使用 layout.shapes)
workspace_shapes = []
for obs in obstacles:
    workspace_shapes.append(
        go.layout.Shape(
            type="circle",
            xref="x1", yref="y1",
            x0=obs['center'][0] - obs['r'],
            y0=obs['center'][1] - obs['r'],
            x1=obs['center'][0] + obs['r'],
            y1=obs['center'][1] + obs['r'],
            fillcolor="Gray",
            line_color="Gray",
            opacity=0.7
        )
    )

# === 子图 2: C-Space ===

# a. 添加 C-Space 障碍物 (使用等高线图)
fig.add_trace(
    go.Contour(
        z=c_space_grid,
        x=theta_range_deg,
        y=theta_range_deg,
        colorscale='Reds',
        showscale=False,
        contours=dict(
            start=0.5,
            end=1,
            size=1
        ),
        name='C-Obstacle'
    ),
    row=1, col=2
)

# b. 添加 C-Space 当前位置标记
fig.add_trace(
    go.Scatter(
        x=[0],  # 初始 t1
        y=[0],  # 初始 t2
        mode='markers',
        marker=dict(color='blue', size=12, symbol='x'),
        name='Current Config'
    ),
    row=1, col=2
)

# --- 6. 更新图表布局 ---

fig.update_layout(
    title_text="2-DOF 机器人交互式 C-Space 可视化",
    width=1000,
    height=500,
    shapes=workspace_shapes, # 应用障碍物
    showlegend=False
)

# 更新 x/y 轴
# 工作空间 (锁定 1:1 比例)
fig.update_xaxes(title_text="X (m)", range=map_range, row=1, col=1)
fig.update_yaxes(title_text="Y (m)", range=map_range, scaleanchor="x1", row=1, col=1)

# C-Space
fig.update_xaxes(title_text="Theta 1 (度)", range=[-180, 180], row=1, col=2)
fig.update_yaxes(title_text="Theta 2 (度)", range=[-180, 180], row=1, col=2)


# --- 7. 创建 ipywidgets 交互 ---

# 将 Figure 包装成 FigureWidget
g = go.FigureWidget(fig)

# 定义滑块
theta1_slider = widgets.FloatSlider(
    min=-180, max=180, step=1, value=0,
    description='Theta 1 (θ₁):',
    continuous_update=True,
    layout={'width': '400px'}
)
theta2_slider = widgets.FloatSlider(
    min=-180, max=180, step=1, value=0,
    description='Theta 2 (θ₂):',
    continuous_update=True,
    layout={'width': '400px'}
)

# 定义更新函数
def update_plot(theta1_deg, theta2_deg):
    
    # 1. 转换单位
    t1_rad = np.deg2rad(theta1_deg)
    t2_rad = np.deg2rad(theta2_deg)
    
    # 2. 计算新的 FK
    P0, P1, P2 = forward_kinematics(t1_rad, t2_rad)
    
    robot_x = [P0[0], P1[0], P2[0]]
    robot_y = [P0[1], P1[1], P2[1]]
    
    # 3. 使用 batch_update() 高效更新图表
    with g.batch_update():
        # 更新图1: 机器人位置 (g.data[0])
        g.data[0].x = robot_x
        g.data[0].y = robot_y
        
        # 更新图2: C-Space 标记位置 (g.data[2])
        g.data[2].x = [theta1_deg]
        g.data[2].y = [theta2_deg]

# --- 8. 链接滑块和图表 ---

# 使用 interact 将滑块连接到 update_plot 函数
interactive_output = widgets.interactive_output(
    update_plot, 
    {'theta1_deg': theta1_slider, 'theta2_deg': theta2_slider}
)

# 组合布局
controls = widgets.VBox([theta1_slider, theta2_slider])

# 显示！
print("控制滑块以移动机器人：")
display(controls, g)

正在计算 C-Space 网格... (可能需要几秒钟)
C-Space 网格计算完成。
控制滑块以移动机器人：


FigureWidget({
    'data': [{'line': {'width': 5},
              'marker': {'size': 10},
              'mode': 'lines+markers',
              'name': 'Robot',
              'type': 'scatter',
              'uid': 'af20eebc-59b7-466d-9840-57cf268fe32f',
              'x': [1.5, 2.5, 3.5],
              'xaxis': 'x',
              'y': [1.5, 1.5, 1.5],
              'yaxis': 'y'},
             {'colorscale': [[0.0, 'rgb(255,245,240)'], [0.125,
                             'rgb(254,224,210)'], [0.25, 'rgb(252,187,161)'],
                             [0.375, 'rgb(252,146,114)'], [0.5, 'rgb(251,106,74)'],
                             [0.625, 'rgb(239,59,44)'], [0.75, 'rgb(203,24,29)'],
                             [0.875, 'rgb(165,15,21)'], [1.0, 'rgb(103,0,13)']],
              'contours': {'end': 1, 'size': 1, 'start': 0.5},
              'name': 'C-Obstacle',
              'showscale': False,
              'type': 'contour',
              'uid': '6d17f8cd-341a-4b54-b368-1453985bcb14',
  

- **正向运动学 (Forward Kinematics, FK)**

正向运动学 (FK) 是指给定一组关节构型 $q$，计算机器人末端执行器（End-Effector）在笛卡尔空间中的位姿（位置和姿态）$T$ 的过程。这个映射函数是唯一且明确的：$T = f(q)$。传统上，Denavit-Hartenberg (DH) 参数法是一种经典的建模方式，而现代方法，如指数积（Product of Exponentials, PoE）公式，因其简洁性和通用性而越来越受欢迎。

- **逆运动学 (Inverse Kinematics, IK)**

逆运动学 (IK) 是 FK 的逆问题：给定末端执行器的期望位姿 $T$，求解所有能够达到该位姿的关节构型 $q$。即求解 $q = f^{-1}(T)$。与 FK 不同，IK 是一个极具挑战性的非线性问题，其核心挑战在于：

    - 解的存在性：并非所有期望位姿都是可达的（例如，超出机器人工作空间）。
    - 解的唯一性：对于非冗余机器人，可能存在多个离散解（例如，“肘部向上”和“肘部向下”）。
    - 解的多样性：对于冗余机器人（自由度 > 6），通常存在无限个解，形成一个连续的解空间。